# Análisis de calidad — pares .npy E3

Analiza los pares `(completo, roto)` ya generados en Drive para detectar modelos problemáticos.

**Checks que hace:**
1. **Degenerada (PCA):** ratio eigenvalor mínimo/máximo < 0.02 → forma plana o lineal
2. **Desconectada (DBSCAN):** la nube se parte en 2+ trozos → asa suelta, artefacto flotante
3. **Fracción eliminada:** fuera del rango esperado 5%–60% → rotura trivial o casi vacía
4. **Desalineación centroide:** centroide de roto muy lejos del de completo (> 0.5 unidades)
5. **Radio anómalo:** el completo no está normalizado a radio ≈ 1

**Orden:**
1. Celda 1 — montar Drive
2. Celda 2 — configurar qué carpeta analizar
3. Celda 3 — escaneo (usa caché si ya existe)
4. Celda 4 — resumen y distribuciones
5. Celda 5 — visualizar los peores casos

In [ ]:
# ── CELDA 1: Montar Drive ──────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive montado.')

In [ ]:

# ── CELDA 2: CONFIGURACIÓN ─────────────────────────────────────
from pathlib import Path

DRIVE = '/content/drive/MyDrive'

# ── elige el dataset a analizar:
CARPETA_DATOS = f'{DRIVE}/Datos_E2_E3/General/sintetico_roturas_v2'
# CARPETA_DATOS = f'{DRIVE}/Datos_E2_E3/General/sintetico_roturas_centradas'

dest = Path(CARPETA_DATOS)

# ── diagnóstico: ver qué hay en la carpeta padre ──────────────
padre = dest.parent
print(f'Carpeta padre: {padre}')
print(f'Subcarpetas y archivos:')
if padre.exists():
    for item in sorted(padre.iterdir()):
        n = len(list(item.glob('*.npy'))) if item.is_dir() else 0
        print(f'  {"D" if item.is_dir() else "F"}  {item.name}  {"("+str(n)+" .npy)" if n else ""}')
else:
    print('  [ERROR] La carpeta padre no existe. Revisa la ruta DRIVE.')

# ── contar pares ──────────────────────────────────────────────
pares = sorted(dest.glob('*_completo.npy')) if dest.exists() else []
print(f'\nDataset : {dest.name}')
print(f'Existe  : {dest.exists()}')
print(f'Pares   : {len(pares)}')
if not pares:
    print('\n[AVISO] No se encontraron pares. Ajusta CARPETA_DATOS con el nombre correcto que aparece arriba.')

# caché en la misma carpeta del dataset
RUTA_CACHE = str(dest / f'analisis_calidad.csv')
print(f'Caché   : {RUTA_CACHE}')


In [ ]:

# ── CELDA 3: ESCANEO (usa caché si ya existe) ──────────────────
import csv
import numpy as np
from pathlib import Path
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors

# re-leer pares por si se ejecuta sin haber pasado por Celda 2 correctamente
dest  = Path(CARPETA_DATOS)
pares = sorted(dest.glob('*_completo.npy'))
print(f'Pares encontrados: {len(pares)}')
if not pares:
    raise RuntimeError('No hay pares. Vuelve a la Celda 2 y corrige CARPETA_DATOS.')

# ── umbrales ──────────────────────────────────────────────────
RATIO_PCA_MIN     = 0.02
FACTOR_EPS_DBSCAN = 4.0
K_VECINOS         = 5
MIN_SAMPLES       = 10
FRAC_MIN          = 0.05
FRAC_MAX          = 0.60
RADIO_MAX         = 2.0
DESALIN_MAX       = 0.5

CAMPOS = ['id', 'dataset', 'problema', 'detalle',
          'ratio_pca', 'n_clusters', 'frac_eliminada',
          'dist_centroides', 'radio_completo']

def analizar_par(ruta_c):
    ruta_r = Path(str(ruta_c).replace('_completo.npy', '_roto.npy'))
    nombre  = ruta_c.stem.replace('_completo', '')
    dataset = 'shapenet' if nombre.startswith('shapenet') else 'objaverse'

    try:
        c = np.load(ruta_c).astype(np.float32)
        r = np.load(ruta_r).astype(np.float32)
    except Exception as e:
        return {'id': nombre, 'dataset': dataset, 'problema': 'ilegible',
                'detalle': str(e), 'ratio_pca': -1, 'n_clusters': -1,
                'frac_eliminada': -1, 'dist_centroides': -1, 'radio_completo': -1}

    problemas, detalles = [], []

    # radio
    radio = float(np.linalg.norm(c, axis=1).max())
    if radio > RADIO_MAX:
        problemas.append('radio_anomalo'); detalles.append(f'radio={radio:.2f}')

    # PCA
    c_c  = c - c.mean(axis=0)
    cov  = np.cov(c_c.T)
    vals = np.linalg.eigvalsh(cov)
    ratio_pca = float(vals[0] / vals[-1]) if vals[-1] > 0 else 0.0
    if ratio_pca < RATIO_PCA_MIN:
        problemas.append('degenerada'); detalles.append(f'ratio_pca={ratio_pca:.4f}')

    # DBSCAN
    nn = NearestNeighbors(n_neighbors=K_VECINOS + 1).fit(c)
    dist, _ = nn.kneighbors(c)
    dist_v = dist[:, 1:][dist[:, 1:] > 0]
    eps = FACTOR_EPS_DBSCAN * float(np.median(dist_v)) if len(dist_v) > 0 else 0.05
    etiquetas = DBSCAN(eps=eps, min_samples=MIN_SAMPLES).fit_predict(c)
    etiq_v = etiquetas[etiquetas >= 0]
    if len(etiq_v) > 0:
        _, conteos = np.unique(etiq_v, return_counts=True)
        n_clusters = int(np.sum(conteos >= 0.05 * len(c)))
    else:
        n_clusters = 0
    if n_clusters >= 2:
        problemas.append('desconectada'); detalles.append(f'{n_clusters}_clusters')

    # fracción eliminada (por volumen de bounding box)
    vol_c = float(np.prod(c.max(axis=0) - c.min(axis=0) + 1e-6))
    vol_r = float(np.prod(r.max(axis=0) - r.min(axis=0) + 1e-6))
    frac  = float(max(0.0, min(1.0, 1.0 - vol_r / vol_c))) if vol_c > 0 else 0.0
    if frac < FRAC_MIN or frac > FRAC_MAX:
        problemas.append('fraccion_anomala'); detalles.append(f'frac={frac:.2f}')

    # desalineación centroide
    dist_centr = float(np.linalg.norm(c.mean(axis=0) - r.mean(axis=0)))
    if dist_centr > DESALIN_MAX:
        problemas.append('centroide_desalineado'); detalles.append(f'dist={dist_centr:.3f}')

    return {
        'id': nombre, 'dataset': dataset,
        'problema': '|'.join(problemas) if problemas else 'ok',
        'detalle':  '|'.join(detalles)  if detalles  else '',
        'ratio_pca': round(ratio_pca, 4),
        'n_clusters': n_clusters,
        'frac_eliminada': round(frac, 3),
        'dist_centroides': round(dist_centr, 3),
        'radio_completo': round(radio, 3),
    }

# ── cargar caché o escanear ───────────────────────────────────
cache_path = Path(RUTA_CACHE)
if cache_path.exists():
    print(f'Cargando caché: {cache_path.name}')
    resultados = []
    with open(cache_path, encoding='utf-8') as f:
        for fila in csv.DictReader(f):
            resultados.append(fila)
    print(f'Cargados {len(resultados)} registros.')
else:
    print(f'Escaneando {len(pares)} pares (puede tardar 5-10 min)...')
    resultados = []
    for i, ruta_c in enumerate(pares):
        resultados.append(analizar_par(ruta_c))
        if (i + 1) % 200 == 0:
            print(f'  {i+1}/{len(pares)} analizados...')

    with open(cache_path, 'w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=CAMPOS)
        w.writeheader()
        w.writerows(resultados)
    print(f'\nGuardado en: {cache_path}')

print(f'Total analizados: {len(resultados)}')


In [ ]:
# ── CELDA 4: RESUMEN Y DISTRIBUCIONES ─────────────────────────
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

ok       = [r for r in resultados if r['problema'] == 'ok']
malos    = [r for r in resultados if r['problema'] != 'ok']
total    = len(resultados)

print(f'=== RESUMEN ===')
print(f'Total pares   : {total}')
print(f'OK            : {len(ok)}  ({100*len(ok)/total:.1f}%)')
print(f'Con problemas : {len(malos)}  ({100*len(malos)/total:.1f}%)')
print()

# contar tipos de problema
tipos = Counter()
for r in malos:
    for p in r['problema'].split('|'):
        tipos[p] += 1

print('Desglose por tipo:')
for tipo, n in tipos.most_common():
    print(f'  {tipo:30s}: {n:4d}  ({100*n/total:.1f}%)')
print()

# desglose por dataset
for ds in ('shapenet', 'objaverse'):
    sub = [r for r in resultados if r['dataset'] == ds]
    sub_malos = [r for r in sub if r['problema'] != 'ok']
    if sub:
        print(f'{ds}: {len(sub_malos)}/{len(sub)} malos ({100*len(sub_malos)/len(sub):.1f}%)')

print()

# ── distribuciones ────────────────────────────────────────────
ratio_pca   = [float(r['ratio_pca'])        for r in resultados if float(r['ratio_pca']) >= 0]
fracs       = [float(r['frac_eliminada'])   for r in resultados if float(r['frac_eliminada']) >= 0]
dist_centr  = [float(r['dist_centroides'])  for r in resultados if float(r['dist_centroides']) >= 0]
radios      = [float(r['radio_completo'])   for r in resultados if float(r['radio_completo']) >= 0]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle(f'Distribuciones de calidad — {Path(CARPETA_DATOS).name}', fontsize=13)

axes[0,0].hist(ratio_pca, bins=50, color='steelblue', edgecolor='white')
axes[0,0].axvline(0.02, color='red', linestyle='--', label='umbral 0.02')
axes[0,0].set_title('Ratio PCA (degeneración)')
axes[0,0].set_xlabel('ratio eigenvalor min/max'); axes[0,0].legend()

axes[0,1].hist(fracs, bins=50, color='steelblue', edgecolor='white')
axes[0,1].axvline(0.05, color='red', linestyle='--', label='límite inf 5%')
axes[0,1].axvline(0.60, color='orange', linestyle='--', label='límite sup 60%')
axes[0,1].set_title('Fracción eliminada estimada')
axes[0,1].set_xlabel('fracción volumen eliminada'); axes[0,1].legend()

axes[1,0].hist(dist_centr, bins=50, color='steelblue', edgecolor='white')
axes[1,0].axvline(0.5, color='red', linestyle='--', label='umbral 0.5')
axes[1,0].set_title('Desalineación centroide roto vs completo')
axes[1,0].set_xlabel('distancia L2'); axes[1,0].legend()

axes[1,1].hist(radios, bins=50, color='steelblue', edgecolor='white')
axes[1,1].axvline(1.0, color='green', linestyle='--', label='radio ideal')
axes[1,1].axvline(2.0, color='red', linestyle='--', label='umbral anomalía')
axes[1,1].set_title('Radio máximo completo (normalización)')
axes[1,1].set_xlabel('radio'); axes[1,1].legend()

plt.tight_layout()
plt.savefig(f'/content/drive/MyDrive/Datos_E2_E3/General/calidad_{Path(CARPETA_DATOS).name}.png', dpi=120)
plt.show()
print('Gráfico guardado en Drive.')

In [ ]:
# ── CELDA 5: VISUALIZAR LOS PEORES CASOS ──────────────────────
# Muestra los N pares con más problemas para inspeccionarlos visualmente.

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

N_MOSTRAR = 8  # cambia este número si quieres ver más o menos

# ordenar por número de problemas y luego por ratio_pca ascendente (más degenerados primero)
malos_ordenados = sorted(
    malos,
    key=lambda r: (-len(r['problema'].split('|')), float(r['ratio_pca']))
)

dest = Path(CARPETA_DATOS)

for i, r in enumerate(malos_ordenados[:N_MOSTRAR]):
    ruta_c = dest / f"{r['id']}_completo.npy"
    ruta_r = dest / f"{r['id']}_roto.npy"
    if not ruta_c.exists():
        print(f'No encontrado: {ruta_c}')
        continue

    c = np.load(ruta_c)
    r_pts = np.load(ruta_r)

    fig = plt.figure(figsize=(12, 5))

    ax1 = fig.add_subplot(1, 2, 1, projection='3d')
    ax1.scatter(c[:, 0], c[:, 1], c[:, 2], s=1, alpha=0.5, color='tab:green')
    ax1.set_title('Completo')
    ax1.set_xlim(-1.5, 1.5); ax1.set_ylim(-1.5, 1.5); ax1.set_zlim(-1.5, 1.5)

    ax2 = fig.add_subplot(1, 2, 2, projection='3d')
    ax2.scatter(c[:, 0], c[:, 1], c[:, 2], s=1, alpha=0.15, color='tab:green', label='GT')
    ax2.scatter(r_pts[:, 0], r_pts[:, 1], r_pts[:, 2], s=1, alpha=0.7, color='tab:red', label='Roto')
    ax2.set_title('Roto sobre GT')
    ax2.legend(loc='upper left', fontsize=7)
    ax2.set_xlim(-1.5, 1.5); ax2.set_ylim(-1.5, 1.5); ax2.set_zlim(-1.5, 1.5)

    plt.suptitle(
        f"[{i+1}/{N_MOSTRAR}] {r['id']}\n"
        f"Problema: {r['problema']}  |  {r['detalle']}",
        fontsize=9
    )
    plt.tight_layout()
    plt.show()

In [ ]:
# ── CELDA 6 (opcional): LISTA DE IDs A DESCARTAR ──────────────
# Imprime la lista de IDs problemáticos por tipo.
# Útil para pasársela al script de entrenamiento como blacklist.

from pathlib import Path

print(f'IDs con problemas ({len(malos)} total):\n')

for tipo in ['degenerada', 'desconectada', 'fraccion_anomala', 'centroide_desalineado', 'radio_anomalo']:
    sub = [r['id'] for r in malos if tipo in r['problema']]
    if sub:
        print(f'--- {tipo} ({len(sub)}) ---')
        for id_ in sub[:20]:  # solo los primeros 20
            print(f'  {id_}')
        if len(sub) > 20:
            print(f'  ... y {len(sub)-20} más')
        print()

# guardar lista completa de malos en Drive
ruta_blacklist = f'/content/drive/MyDrive/Datos_E2_E3/General/blacklist_{Path(CARPETA_DATOS).name}.txt'
with open(ruta_blacklist, 'w') as f:
    for r in malos:
        f.write(r['id'] + '\n')
print(f'Blacklist guardada en: {ruta_blacklist}')